## 0. 환경설정 - 메이크업 챗봇

In [ ]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python gradio python-dotenv -q

In [ ]:
import os, json, re, time
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 환경변수 로드

from neo4j import GraphDatabase, basic_auth
from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG
from openai import OpenAI
import gradio as gr

print("모듈 로드 완료")

In [ ]:
# OPENAI_API_KEY는 .env에서 자동 로드됨
client = OpenAI()

## 1. 데이터 로드 - 메이크업

In [ ]:
_BASE = os.path.abspath(os.getcwd())
DATA_FILE = os.path.join(_BASE, "json", "iwedding_makeup_detail.json")

with open(DATA_FILE, "r", encoding="utf-8") as f:
    raw = json.load(f)

# JSON 구조에 따라 리스트 추출
if isinstance(raw, list):
    all_data = raw
else:
    for key in ["data", "items", "result", "partners"]:
        if key in raw and isinstance(raw[key], list):
            all_data = raw[key]
            break
    else:
        all_data = list(raw.values())[0] if raw else []

print(f"메이크업: {len(all_data)}개 로드")
prices = [d.get("salePrice",0) for d in all_data if d.get("salePrice",0) > 0]
if prices:
    print(f"가격 범위: {min(prices):,}원 ~ {max(prices):,}원 (평균 {sum(prices)//len(prices):,}원)")
print(f"업체 수: {len(set(d.get('name','') for d in all_data))}개")

## 2. Neo4j 연결 (읽기 전용 — 데이터 적재는 db_load.py)

In [ ]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PW", "password123")

driver = GraphDatabase.driver(NEO4J_URI, auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    cnt = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    print(f"Neo4j 현재 노드 수: {cnt}")

In [ ]:
# 데이터 존재 여부 확인 (읽기 전용 — DB 삭제/삽입은 db_load.py에서만 수행)
with driver.session() as session:
    result = session.run(
        "MATCH (v:Vendor {category: $cat}) RETURN count(v) AS cnt",
        cat="makeup"
    ).single()
    vendor_cnt = result["cnt"]
    if vendor_cnt == 0:
        print("⚠ makeup Vendor가 0개입니다. 먼저 db_load.py를 실행하세요.")
    else:
        print(f"makeup Vendor {vendor_cnt}개 확인 — 정상")

In [ ]:
# 관계 통계 확인
with driver.session() as session:
    rows = session.run("""
        MATCH (v:Vendor {category: 'makeup'})-[r]->()
        RETURN type(r) AS rel, count(r) AS cnt
        ORDER BY cnt DESC
    """).data()
    for row in rows:
        print(f"  {row['rel']}: {row['cnt']}개")
    if not rows:
        print("⚠ 관계가 없습니다. db_load.py를 실행하세요.")

## 3. MySQL (더미 fallback)

In [ ]:
import mysql.connector

MYSQL_HOST = os.environ.get("MYSQL_HOST", "")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
COUPLE_ID = 2

conn = None
if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST, user=MYSQL_USER,
            password=MYSQL_PASSWORD, database=MYSQL_DB,
            port=int(os.environ.get("MYSQL_PORT", 3306))
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 - 더미 사용")

def get_user_preference(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_preferences WHERE couple_id = %s", (couple_id,))
            row = cur.fetchone(); cur.close()
            return row
        except: pass
    return {'couple_id': 2, 'region': '서울', 'sub_region': '강남구', 'style': '내추럴', 'budget': '70만원', 'category': '메이크업'}

def get_user_likes(conn, couple_id):
    if conn:
        try:
            cur = conn.cursor(dictionary=True)
            cur.execute("SELECT * FROM couple_venue_likes WHERE couple_id = %s", (couple_id,))
            rows = cur.fetchall(); cur.close()
            return rows
        except: pass
    return [{"partner_id": all_data[0]["partnerId"], "name": all_data[0]["name"], "category": "메이크업"}]

print("사용자 함수 정의 완료")

## 4. Neo4j 스키마 추출

In [ ]:
def get_schema(uri, user, password):
    d = GraphDatabase.driver(uri, auth=basic_auth(user, password))
    with d.session() as session:
        nodes = session.run("MATCH (n) WITH DISTINCT labels(n) AS lbls, keys(n) AS ks, n UNWIND lbls AS l UNWIND ks AS k RETURN l, k, n[k] AS sv")
        rels = session.run("MATCH (a)-[r]->(b) RETURN DISTINCT labels(a) AS sl, type(r) AS rt, labels(b) AS el")
        schema = {"nodes": {}, "relations": []}
        for rec in nodes:
            l, k = rec["l"], rec["k"]
            if l not in schema["nodes"]: schema["nodes"][l] = {}
            v = rec["sv"]
            t = "STRING" if isinstance(v, str) else "INTEGER" if isinstance(v, int) else "FLOAT" if isinstance(v, float) else "UNKNOWN"
            schema["nodes"][l][k] = t
        for rec in rels:
            sl = rec["sl"][0] if rec["sl"] else "?"
            el = rec["el"][0] if rec["el"] else "?"
            schema["relations"].append(f"(:{sl})-[:{rec['rt']}]->(:{el})")
    d.close()
    return schema

def format_schema(schema):
    lines = ["Node properties:"]
    for label, props in schema["nodes"].items():
        ps = ", ".join(f"{k}: {v}" for k, v in props.items())
        lines.append(f"  {label} {{{ps}}}")
    lines.append("Relationships:")
    for r in schema["relations"]: lines.append(f"  {r}")
    return "\n".join(lines)

schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
neo4j_schema = format_schema(schema)
print(neo4j_schema)


## 5. Few-shot Cypher 예시

In [ ]:
examples = [
    """USER INPUT: '메이크업 추천해줘'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '헤어메이크업 알아보고 있어'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '괜찮은 메이크업샵 있을까?'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0 AND v.rating >= 90
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '평점 좋은 메이크업샵'
QUERY:
MATCH (v:Vendor) WHERE v.rating > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC, v.reviewCnt DESC LIMIT 10""",

    """USER INPUT: '리뷰 많은 메이크업샵'
QUERY:
MATCH (v:Vendor) WHERE v.reviewCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.reviewCnt DESC, v.rating DESC LIMIT 10""",

    """USER INPUT: '50만원 이하 메이크업 추천'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice <= 500000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '30만원에서 50만원 사이 메이크업'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice >= 300000 AND v.salePrice <= 500000
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '가장 저렴한 메이크업 5개'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.profileUrl AS url
ORDER BY v.salePrice ASC LIMIT 5""",

    """USER INPUT: '고급 프리미엄 메이크업'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice >= 1000000
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '할인 많이 된 메이크업'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice > 0 AND v.productPrice > v.salePrice
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, (v.productPrice - v.salePrice) AS discount, v.rating AS rating, v.profileUrl AS url
ORDER BY discount DESC LIMIT 10""",

    """USER INPUT: '너무 비싸지 않은 적당한 메이크업'
QUERY:
MATCH (v:Vendor) WHERE v.salePrice <= 700000 AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '내추럴 메이크업 추천해줘'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag) OPTIONAL MATCH (v)-[:HAS_STYLE]->(sf:StyleFilter)
WHERE t.name CONTAINS '내추럴' OR sf.name CONTAINS '내추럴' OR t.name CONTAINS '깨끗'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '음영 메이크업 스타일'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '음영'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '깨끗하고 화사한 스타일'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '깨끗' OR t.name CONTAINS '화사'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '실장급 메이크업 추천'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '실장' OR v.productName CONTAINS '실장' OR v.detailCmt CONTAINS '실장'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '원장님이 직접 해주는 곳'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '원장' OR t.name CONTAINS '대표' OR v.productName CONTAINS '원장' OR v.productName CONTAINS '대표'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '촬영+본식 메이크업 추천'
QUERY:
MATCH (v:Vendor) WHERE (v.productName CONTAINS '촬영' AND v.productName CONTAINS '본식') OR v.detailCmt CONTAINS '촬영+본식'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '본식만 하는 메이크업'
QUERY:
MATCH (v:Vendor) WHERE v.productName CONTAINS '본식' AND NOT v.productName CONTAINS '촬영'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '50만원 이하 내추럴 메이크업'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE v.salePrice <= 500000 AND v.salePrice > 0 AND (t.name CONTAINS '내추럴' OR t.name CONTAINS '깨끗')
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '강남에서 실장급 메이크업'
QUERY:
MATCH (v:Vendor)-[:IN_REGION]->(r:Region) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE r.name CONTAINS '강남' AND (t.name CONTAINS '실장' OR v.productName CONTAINS '실장')
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '촬영 말고 본식 메이크업만'
QUERY:
MATCH (v:Vendor) WHERE v.productName CONTAINS '본식' AND NOT v.productName CONTAINS '촬영' AND v.salePrice > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '음영 말고 내추럴로'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WITH v, collect(DISTINCT t.name) AS tags
WHERE any(tag IN tags WHERE tag CONTAINS '내추럴' OR tag CONTAINS '깨끗') AND NONE(tag IN tags WHERE tag CONTAINS '음영')
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '르보청담이랑 치치라보 비교해줘'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '르보' OR v.name CONTAINS '치치라보'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.detailCmt AS description, v.packageInfoStr AS packageInfo, collect(DISTINCT t.name) AS tags""",

    """USER INPUT: '고센뷰티랑 로나 뭐가 나아?'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '고센' OR v.name CONTAINS '로나'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.productPrice AS originalPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.detailCmt AS description, v.packageInfoStr AS packageInfo, collect(DISTINCT t.name) AS tags""",

    """USER INPUT: '르보청담 가격 알려줘'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '르보'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.name AS name, v.productPrice AS originalPrice, v.salePrice AS salePrice, v.eventPrice AS eventPrice, v.rating AS rating, v.reviewCnt AS reviewCnt, v.address AS address, v.tel AS tel, v.detailCmt AS description, v.packageInfoStr AS packageInfo, v.addcostStr AS addcost, v.reviewsStr AS reviews, collect(DISTINCT t.name) AS tags""",

    """USER INPUT: '꾸띠원 패키지 구성은?'
QUERY:
MATCH (v:Vendor) WHERE v.name CONTAINS '꾸띠원'
OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
RETURN v.name AS name, v.salePrice AS salePrice, v.rating AS rating, v.detailCmt AS description, v.packageInfoStr AS packageInfo, v.addcostStr AS addcost, collect(DISTINCT t.name) AS tags""",

    """USER INPUT: '요즘 인기있는 메이크업샵'
QUERY:
MATCH (v:Vendor) WHERE v.orderCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.reviewCnt AS reviewCnt, v.orderCnt AS orders, v.likeCnt AS likes, v.profileUrl AS url
ORDER BY v.orderCnt DESC, v.likeCnt DESC LIMIT 10""",

    """USER INPUT: '찜 많이 받은 메이크업샵'
QUERY:
MATCH (v:Vendor) WHERE v.likeCnt > 0
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.likeCnt AS likes, v.profileUrl AS url
ORDER BY v.likeCnt DESC LIMIT 10""",

    """USER INPUT: '신랑 메이크업도 해주는 곳'
QUERY:
MATCH (v:Vendor) WHERE v.productName CONTAINS '신랑' OR v.detailCmt CONTAINS '신랑'
RETURN v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",

    """USER INPUT: '헤어변형 포함된 메이크업'
QUERY:
MATCH (v:Vendor) OPTIONAL MATCH (v)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '헤어변형' OR v.detailCmt CONTAINS '헤어변형'
RETURN DISTINCT v.partnerId AS id, v.name AS name, v.salePrice AS price, v.rating AS rating, v.detailCmt AS description, v.profileUrl AS url
ORDER BY v.rating DESC LIMIT 10""",
]

print(f"Few-shot 예시 {len(examples)}개 로드 완료")


## 6. LLM / Retriever / GraphRAG

In [ ]:
llm = OpenAILLM(model_name="gpt-4o", model_params={"temperature": 0, "max_tokens": 2000})
retriever = Text2CypherRetriever(driver=driver, llm=llm, neo4j_schema=neo4j_schema, examples=examples)
rag = GraphRAG(retriever=retriever, llm=llm)
print("GraphRAG 초기화 완료")

## 7. Gradio 챗봇

In [ ]:
def response_fn(message, chat_history):
    chat_history = chat_history or []
    msg = message.strip()

    if any(x in msg for x in ["내 취향", "내 스타일", "내 선호"]):
        pref = get_user_preference(conn, COUPLE_ID)
        if pref:
            lines = [f"- {k}: {v}" for k, v in pref.items() if k != "couple_id" and v]
            answer = "현재 저장된 취향 정보입니다.\n" + "\n".join(lines)
        else:
            answer = "저장된 취향 정보가 없습니다."
    elif any(x in msg for x in ["좋아요한", "찜한", "찜 목록"]):
        likes = get_user_likes(conn, COUPLE_ID)
        if likes:
            lines = [f"- {l.get('name','알수없음')} ({l.get('category','')})" for l in likes]
            answer = f"좋아요한 업체가 {len(likes)}건 있습니다.\n" + "\n".join(lines)
        else:
            answer = "좋아요한 업체가 없습니다."
    else:
        try:
            result = rag.search(query_text=message, retriever_config={"top_k": 10})
            answer = result.answer
        except Exception as e:
            print(f"[GraphRAG 오류] {e}")
            answer = "죄송합니다. 처리 중 오류가 발생했습니다. 다시 시도해주세요."

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": answer})
    return "", chat_history


with gr.Blocks(title="메이크업 추천 챗봇") as demo:
    gr.Markdown("# 메이크업 추천 챗봇\n웨딩 메이크업 추천을 도와드립니다! (서울/경기 59개 업체)")
    chatbot = gr.Chatbot(height=600, type="messages")
    with gr.Row():
        msg = gr.Textbox(placeholder="예: 50만원 이하 내추럴 메이크업 추천해줘", show_label=False, scale=9)
        send_btn = gr.Button("전송", scale=1)
    gr.Markdown("### 이런 질문을 해보세요")
    with gr.Row():
        gr.Button("메이크업 추천해줘").click(lambda: ("메이크업 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("50만원 이하").click(lambda: ("50만원 이하 메이크업 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("내추럴 스타일").click(lambda: ("내추럴 스타일 메이크업 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("실장급 추천").click(lambda: ("실장급 메이크업 추천해줘", None), outputs=[msg, chatbot])
    with gr.Row():
        gr.Button("촬영+본식 세트").click(lambda: ("촬영+본식 메이크업 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("리뷰 좋은 곳").click(lambda: ("리뷰 좋은 메이크업 추천해줘", None), outputs=[msg, chatbot])
        gr.Button("저렴한 순").click(lambda: ("가장 저렴한 메이크업 5개 알려줘", None), outputs=[msg, chatbot])
        gr.Button("르보청담 상세").click(lambda: ("르보청담 가격이랑 구성 알려줘", None), outputs=[msg, chatbot])
    msg.submit(response_fn, [msg, chatbot], [msg, chatbot])
    send_btn.click(response_fn, [msg, chatbot], [msg, chatbot])

demo.launch(share=True)